## Final Project Business Intelligence   

**Group**: Castillo, Ignacia; Lara, Josefa; Lineros Pablo; Matosas, María Emilia; Toledo, Catalina.    
**Turn-in date**: 11th of august 2026    
**Instructor**: Christopher Castro Araya.    

#### BI Pipeline: The notebook will have the following structure:
 
- Frame & KPIs: Business question, decision-maker, success metric and KPIs.
- Prepare data: EDA, cleaning, missing/outliers, encoding, features, no leakage.
- Model & evaluate: Prediction or segmentation, proper validation, honest metrics.
- Communicate: Dashboard on PowerBI (not here) + a recommendation to act on.
- Ethics & limits: Bias, fairness, privacy, and honest limitations of your result.

#### About the dataset:
We chose an apple products pricing dataset found on Kaggle Datasets that goes from 2020 to 2026. It has 80001 Rows and 14 columns of which we will be using 12.     
The columnswe will use are: 
- Date 
- Platform 
- Product_Category 
- Model_Name 
- Condition 
- Launch_Price_USD  
- Current_Price_USD  
- Discount_Pct 
- Sale_Event 
- Stock_Status 
- Rating 
- Reviews_Count 

### Frame & KPIs   

- Business question: ¿How have Apple product prices evolved between 2020 and 2026, and what strategic opportunities exist to optimize margins, competitive positioning, and customer segmentation?    

- Decision maker: Vice President of E-Commerce Channel Strategy & Commercial Operations (Retail Channel Lead for Amazon & Flipkart).    

- Success metric: We are trying to generate insights that allow:
    - Maximize net income by adjusting prices and discounts.
    - Improve customer perception (ratings and reviews).
    - Increase the effectiveness of sales campaigns (events, promotions).

- KPIs: 
    - Average price by category and condition.
    - Variation of price vs launching price.
    - Discount percentage.
    - Stock availability ratio.
    - Customer sentiment index.
    - Impact on sell event.
    - Revenue proxy.

### EDA

In [4]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv(r'C:\Users\titim\Desktop\Proyecto BI\apple_products_pricing_2020_2026.csv')

print("Shape:", df.shape)
print("\nColumns and Dtypes:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nHead:")
print(df.head())
print("\nSummary Stats:")
print(df.describe(include='all'))

Shape: (80000, 14)

Columns and Dtypes:
Date                  object
Platform              object
Product_Category      object
Model_Name            object
Condition             object
Launch_Price_USD       int64
Launch_Price_INR       int64
Current_Price_USD    float64
Current_Price_INR    float64
Discount_Pct         float64
Sale_Event            object
Stock_Status          object
Rating               float64
Reviews_Count          int64
dtype: object

Missing values:
Date                     0
Platform                 0
Product_Category         0
Model_Name               0
Condition                0
Launch_Price_USD         0
Launch_Price_INR         0
Current_Price_USD        0
Current_Price_INR        0
Discount_Pct             0
Sale_Event           73351
Stock_Status             0
Rating                   0
Reviews_Count            0
dtype: int64

Head:
         Date  Platform Product_Category                   Model_Name  \
0  2020-09-19  Flipkart            Watch  Apple Watc

In [5]:
# Detailed exploratory calculations
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Sale_Event_Filled'] = df['Sale_Event'].fillna('Regular Days')

# Revenue proxy (commonly estimated in e-commerce as Price * Reviews)
df['Revenue_Proxy_USD'] = df['Current_Price_USD'] * df['Reviews_Count']

# Summary table 1: By Product Category
cat_summary = df.groupby('Product_Category').agg(
    Record_Count=('Model_Name', 'count'),
    Avg_Launch_USD=('Launch_Price_USD', 'mean'),
    Avg_Current_USD=('Current_Price_USD', 'mean'),
    Avg_Discount_Pct=('Discount_Pct', 'mean'),
    Avg_Rating=('Rating', 'mean'),
    Total_Reviews=('Reviews_Count', 'sum'),
    Total_Revenue_Proxy=('Revenue_Proxy_USD', 'sum')
).reset_index()

print("=== Category Summary ===")
print(cat_summary)

# Summary table 2: By Condition
cond_summary = df.groupby('Condition').agg(
    Record_Count=('Model_Name', 'count'),
    Avg_Launch_USD=('Launch_Price_USD', 'mean'),
    Avg_Current_USD=('Current_Price_USD', 'mean'),
    Avg_Discount_Pct=('Discount_Pct', 'mean'),
    Avg_Rating=('Rating', 'mean')
).reset_index()

print("\n=== Condition Summary ===")
print(cond_summary)

# Summary table 3: By Sale Event
event_summary = df.groupby('Sale_Event_Filled').agg(
    Record_Count=('Model_Name', 'count'),
    Avg_Discount_Pct=('Discount_Pct', 'mean'),
    Avg_Current_USD=('Current_Price_USD', 'mean'),
    Avg_Rating=('Rating', 'mean')
).reset_index()

print("\n=== Sale Event Summary ===")
print(event_summary)

# Summary table 4: Stock Status
stock_summary = df.groupby('Stock_Status').agg(
    Record_Count=('Model_Name', 'count'),
    Pct_of_Total=('Model_Name', lambda x: len(x) / len(df) * 100),
    Avg_Discount_Pct=('Discount_Pct', 'mean')
).reset_index()

print("\n=== Stock Status Summary ===")
print(stock_summary)

# Summary table 5: Yearly Price Trend
year_summary = df.groupby('Year').agg(
    Avg_Launch_USD=('Launch_Price_USD', 'mean'),
    Avg_Current_USD=('Current_Price_USD', 'mean'),
    Avg_Discount_Pct=('Discount_Pct', 'mean'),
    Avg_Rating=('Rating', 'mean')
).reset_index()

print("\n=== Yearly Trend ===")
print(year_summary)

=== Category Summary ===
  Product_Category  Record_Count  Avg_Launch_USD  Avg_Current_USD  \
0              Mac         18020     1612.312986      1382.489812   
1            Watch         17865      533.320739       398.638561   
2             iPad         15526      749.823779       573.083382   
3           iPhone         28589      940.687362       758.674718   

   Avg_Discount_Pct  Avg_Rating  Total_Reviews  Total_Revenue_Proxy  
0         15.430627    4.450461       41174963         5.144980e+10  
1         26.266779    4.448066       42520463         1.478418e+10  
2         26.189856    4.449916       43597043         2.105661e+10  
3         19.572790    4.451240       65187205         4.403867e+10  

=== Condition Summary ===
             Condition  Record_Count  Avg_Launch_USD  Avg_Current_USD  \
0                  New         59985      964.676919       830.066030   
1  Renewed/Refurbished         20015      961.807894       641.023114   

   Avg_Discount_Pct  Avg_Rating 

In [6]:
# Platform breakdown
plat_summary = df.groupby('Platform').agg(
    Record_Count=('Model_Name', 'count'),
    Avg_Discount_Pct=('Discount_Pct', 'mean'),
    Avg_Current_USD=('Current_Price_USD', 'mean'),
    Avg_Rating=('Rating', 'mean')
).reset_index()

print("=== Platform Summary ===")
print(plat_summary)

# Negative discount analysis
neg_disc = df[df['Discount_Pct'] < 0]
print(f"\nNegative discount count: {len(neg_disc)} ({len(neg_disc)/len(df)*100:.2f}%)")
print(neg_disc[['Launch_Price_USD', 'Current_Price_USD', 'Discount_Pct']].head())

# Correlations
corr_cols = ['Launch_Price_USD', 'Current_Price_USD', 'Discount_Pct', 'Rating', 'Reviews_Count', 'Revenue_Proxy_USD']
print("\n=== Correlation Matrix ===")
print(df[corr_cols].corr())

=== Platform Summary ===
   Platform  Record_Count  Avg_Discount_Pct  Avg_Current_USD  Avg_Rating
0    Amazon         39957         21.515399       783.968025    4.450802
1  Flipkart         40043         21.322461       781.574259    4.449397

Negative discount count: 8669 (10.84%)
   Launch_Price_USD  Current_Price_USD  Discount_Pct
0               429             435.81          -1.6
1               429             436.49          -1.7
4               429             436.22          -1.7
5               429             436.29          -1.7
6               429             433.06          -0.9

=== Correlation Matrix ===
                   Launch_Price_USD  Current_Price_USD  Discount_Pct  \
Launch_Price_USD           1.000000           0.947179     -0.322066   
Current_Price_USD          0.947179           1.000000     -0.574824   
Discount_Pct              -0.322066          -0.574824      1.000000   
Rating                    -0.001760           0.110571     -0.310257   
Reviews_Co

In [7]:
# Get exact numbers for formatting in table format
print("CATEGORY TABLE:")
for idx, r in cat_summary.iterrows():
    print(f"| {r['Product_Category']} | ${r['Avg_Launch_USD']:.2f} | ${r['Avg_Current_USD']:.2f} | {r['Avg_Discount_Pct']:.1f}% | {r['Avg_Rating']:.2f} | {r['Total_Reviews']:,} | ${r['Total_Revenue_Proxy']/1e9:.2f}B |")

print("\nCONDITION TABLE:")
for idx, r in cond_summary.iterrows():
    print(f"| {r['Condition']} | ${r['Avg_Launch_USD']:.2f} | ${r['Avg_Current_USD']:.2f} | {r['Avg_Discount_Pct']:.1f}% | {r['Avg_Rating']:.2f} | {r['Record_Count']:,} |")

print("\nEVENT TABLE:")
for idx, r in event_summary.iterrows():
    print(f"| {r['Sale_Event_Filled']} | ${r['Avg_Current_USD']:.2f} | {r['Avg_Discount_Pct']:.1f}% | {r['Avg_Rating']:.2f} | {r['Record_Count']:,} |")

print("\nSTOCK TABLE:")
for idx, r in stock_summary.iterrows():
    print(f"| {r['Stock_Status']} | {r['Record_Count']:,} | {r['Pct_of_Total']:.1f}% | {r['Avg_Discount_Pct']:.1f}% |")

print("\nYEARLY TABLE:")
for idx, r in year_summary.iterrows():
    print(f"| {r['Year']} | ${r['Avg_Launch_USD']:.2f} | ${r['Avg_Current_USD']:.2f} | {r['Avg_Discount_Pct']:.1f}% | {r['Avg_Rating']:.2f} |")

CATEGORY TABLE:
| Mac | $1612.31 | $1382.49 | 15.4% | 4.45 | 41,174,963 | $51.45B |
| Watch | $533.32 | $398.64 | 26.3% | 4.45 | 42,520,463 | $14.78B |
| iPad | $749.82 | $573.08 | 26.2% | 4.45 | 43,597,043 | $21.06B |
| iPhone | $940.69 | $758.67 | 19.6% | 4.45 | 65,187,205 | $44.04B |

CONDITION TABLE:
| New | $964.68 | $830.07 | 16.7% | 4.55 | 59,985 |
| Renewed/Refurbished | $961.81 | $641.02 | 35.5% | 4.15 | 20,015 |

EVENT TABLE:
| Big Billion Days | $596.32 | 37.0% | 4.45 | 1,579 |
| Black Friday | $685.29 | 31.5% | 4.45 | 2,497 |
| Great Indian Festival | $644.10 | 33.4% | 4.44 | 1,504 |
| Prime Day | $664.37 | 34.4% | 4.45 | 1,069 |
| Regular Days | $794.67 | 20.3% | 4.45 | 73,351 |

STOCK TABLE:
| In Stock | 55,034 | 68.8% | 18.4% |
| Low Stock | 11,491 | 14.4% | 27.3% |
| Out of Stock | 13,475 | 16.8% | 28.7% |

YEARLY TABLE:
| 2020.0 | $721.50 | $662.20 | 8.7% | 4.45 |
| 2021.0 | $793.45 | $722.25 | 9.2% | 4.45 |
| 2022.0 | $852.71 | $742.50 | 14.2% | 4.45 |
| 2023.0 | $939